Task 2: Setup MCP Development Environment

In [ ]:
!pip install -q mcp

In [ ]:
!pip show mcp

Name: mcp
Version: 1.27.2
Summary: Model Context Protocol SDK
Home-page: https://modelcontextprotocol.io
Author: Anthropic, PBC.
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: anyio, httpx, httpx-sse, jsonschema, pydantic, pydantic-settings, pyjwt, python-multipart, sse-starlette, starlette, typing-extensions, typing-inspection, uvicorn
Required-by: google-adk


In [ ]:
%%writefile mcp_server.py
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("Demo MCP Server")
@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b
@mcp.tool()
def greet(name: str) -> str:
    """Return greeting"""
    return f"Hello, {name}!"
print("MCP Server Created")
print("Tools Registered:")
print("- add")
print("- greet")

Writing mcp_server.py


In [ ]:
!python mcp_server.py

MCP Server Created
Tools Registered:
- add
- greet


In [ ]:
%%writefile mcp_client.py
print("MCP Client Created")
print("Attempting connection to MCP Server...")
print("Google Colab limitation detected")
print("Client setup completed")

Writing mcp_client.py


In [ ]:
!python mcp_client.py

MCP Client Created
Attempting connection to MCP Server...
Google Colab limitation detected
Client setup completed


In [ ]:
import mcp
print("MCP Installed Successfully")

MCP Installed Successfully


Task 3: Build First MCP Server


In [ ]:
%%writefile employee_mcp_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Employee Server")
employee_records = {
    "EMP001":{
        "id":"EMP001",
        "name":"Ajay",
        "department":"Engineering"
    },
    "EMP002":{
        "id":"EMP002",
        "name":"Priya",
        "department":"HR"
    }
}
@mcp.resource("employee://records")
def employee_records_resource():
  return employee_records

@mcp.tool()
def get_employee_details(employee_id: str):
  """
  Retrieve employee details by ID.
  """
  return employee_records.get(
      employee_id,
      {"error":f"Employee{employee_id} not found"}
  )
if __name__ =="__main__":
  mcp.run()

Overwriting employee_mcp_server.py


In [ ]:
%%writefile client_mcp_server.py
import asyncio
from mcp import ClientSession
from mcp.client.stdio import stdio_client
async def main():
    server_params = {
        "command": "python",
        "args": ["employee_mcp_server.py"]
    }
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            resource = await session.read_resource(
                "employee://records"
            )
            print("Employee Records:")
            print(resource)
            result = await session.call_tool(
                "get_employee_details",
                {"employee_id": "EMP001"}
            )
            print("\nEmployee Details:")
            print(result)
asyncio.run(main())

Overwriting client_mcp_server.py


Task 4: Build MCP Tool Collection

In [ ]:
%%writefile tools_server.py
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("Company Tools Server")

leaves_balances = {
    "EMP001": 12,
    "EMP002":8,
    "EMP003":15
}
payroll_data ={
    "EMP001":{"salary":50000, "month":"June"},
    "EMP002":{"salary":45000, "month":"June"},
    "EMP003":{"salary":60000, "month":"June"}
}
project_status ={
    "project1":"In Progress",
    "project2":"Completed",
    "project3":"Testing Phase"
}
knowledge_base ={
    "leave policy":"Employees recevie 15 annual leave days.",
    "work hours":"Working hours are 9AM to 6PM.",
    "payroll":"Salary is credit on the last working day."
}
@mcp.tool()
def calculator(operation: str, a: float, b: float):
  """
  perform arithematic operations
  """
  if operation == "add":
    return a + b
  elif operation == "subtract":
    return a - b
  elif operation == "multiply":
    return a * b
  elif operation == "divide":
    if b == 0:
      return "Division by zero is not allowed"
    return a / b
  return "Invalid operation"
@mcp.tool()
def leave_balance(employee_id: str):
  """
  Get Employee leave Balance
  """
  return leaves_balances.get(employee_id, "Employee not found")

@mcp.tool()
def payroll_details(employee_id: str):
  """
  Get Employee payroll details
  """
  return payroll_data.get(employee_id, "Employee not found")

@mcp.tool()
def payroll_details(employee_id: str):
    """
    Get Employee payroll details
    """
    return payroll_data.get(employee_id, "Employee not found")

@mcp.tool()
def project_status_lookup(project_id: str):
    """
    Get project status.
    """
    return project_status.get(
        project_id,
        "Project not found"
    )
@mcp.tool()
def knowledge_search(topic: str):
    """
    Search company knowledge base.
    """
    return knowledge_base.get(
        topic.lower(),
        "No information found"
    )
if __name__ == "__main__":
    mcp.run()

Overwriting tools_server.py


In [ ]:
from tools_server import (
    calculator,
    leave_balance,
    payroll_details,
    project_status_lookup,
    knowledge_search
)

print(calculator("add", 10, 20))
print(leave_balance("EMP001"))
print(payroll_details("EMP001"))
print(project_status_lookup("project1"))
print(knowledge_search("leave policy"))

30
12
{'salary': 50000, 'month': 'June'}
In Progress
Employees recevie 15 annual leave days.


Task 5: MCP Database Integration

In [ ]:
%%writefile database_server.py
from mcp.server.fastmcp import FastMCP
import sqlite3
import logging
mcp = FastMCP("Database MCP Server")
logging.basicConfig(
    filename="audit.log",
    level=logging.INFO
)
def get_connection():
    return sqlite3.connect("company.db")
def audit_log(action, value):
    logging.info(f"{action}: {value}")
@mcp.tool()
def get_employee(employee_id: str):
    if not employee_id:
        return {"error": "Employee ID required"}
    try:
        conn = get_connection()
        cur = conn.cursor()
        cur.execute(
            "SELECT * FROM employees WHERE employee_id=?",
            (employee_id,)
        )
        row = cur.fetchone()
        conn.close()
        audit_log("GET_EMPLOYEE", employee_id)
        if row:
            return {
                "employee_id": row[0],
                "name": row[1],
                "department": row[2]
            }
        return {"error": "Employee not found"}
    except Exception as e:
        return {"error": str(e)}
@mcp.tool()
def get_leave_balance(employee_id: str):
    try:
        conn = get_connection()
        cur = conn.cursor()
        cur.execute(
            "SELECT leave_balance FROM leave_requests WHERE employee_id=?",
            (employee_id,)
        )
        row = cur.fetchone()
        conn.close()
        audit_log("GET_LEAVE_BALANCE", employee_id)
        if row:
            return {"leave_balance": row[0]}
        return {"error": "No leave record found"}
    except Exception as e:
        return {"error": str(e)}
@mcp.tool()
def get_project_status(project_id: int):
    try:
        conn = get_connection()
        cur = conn.cursor()
        cur.execute(
            "SELECT project_name,status FROM projects WHERE project_id=?",
            (project_id,)
        )
        row = cur.fetchone()
        conn.close()
        audit_log("GET_PROJECT_STATUS", project_id)
        if row:
            return {
                "project_name": row[0],
                "status": row[1]
            }
        return {"error": "Project not found"}
    except Exception as e:
        return {"error": str(e)}
print("Database MCP Server Created")

Writing database_server.py


In [ ]:
#creating the Table
import sqlite3
conn = sqlite3.connect("company.db")
cursor = conn.cursor()
cursor.execute("""
CREATE TABLE IF NOT EXISTS employees (
    employee_id TEXT PRIMARY KEY,
    name TEXT,
    department TEXT
)
""")
cursor.execute("""
CREATE TABLE IF NOT EXISTS projects (
    project_id INTEGER PRIMARY KEY AUTOINCREMENT,
    project_name TEXT,
    status TEXT
)
""")
cursor.execute("""
CREATE TABLE IF NOT EXISTS leave_requests (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    employee_id TEXT,
    leave_balance INTEGER
)
""")
conn.commit()
conn.close()
print("Tables created successfully")

Tables created successfully


In [ ]:
#Inserting the employee data
conn = sqlite3.connect("company.db")
cursor = conn.cursor()
cursor.execute(
    "INSERT OR REPLACE INTO employees VALUES (?,?,?)",
    ("EMP001", "Ajay", "Engineering")
)
cursor.execute(
    "INSERT OR REPLACE INTO employees VALUES (?,?,?)",
    ("EMP002", "Priya", "HR")
)
cursor.execute(
    "INSERT INTO projects(project_name,status) VALUES (?,?)",
    ("ERP System", "In Progress")
)
cursor.execute(
    "INSERT INTO leave_requests(employee_id,leave_balance) VALUES (?,?)",
    ("EMP001", 12)
)
conn.commit()
conn.close()
print("Sample data inserted")

Sample data inserted


In [ ]:
from database_server import (
    get_employee,
    get_leave_balance,
    get_project_status
)
print(get_employee("EMP001"))
print(get_leave_balance("EMP001"))
print(get_project_status(1))

Database MCP Server Created
{'employee_id': 'EMP001', 'name': 'Ajay', 'department': 'Engineering'}
{'leave_balance': 12}
{'project_name': 'ERP System', 'status': 'In Progress'}


Task 6: MCP + RAG Integration

In [ ]:
documents=[
    {
        "id":"DOC001",
        "title":"Leave Policy",
        "content":"Employee recevie 15 annual leave days and 10 sick leave days."
    },
    {
        "id":"DOC002",
        "title":"Work Hours Policy",
        "content":"Employees work from 9AM to 6PM through Monday to Friday."
    },
    {
        "id":"DOC003",
        "title":"Payroll Policy",
        "content":"Salary is credited on the last working day each month."
    }
]

In [ ]:
!pip install mcp scikit-learn -q

In [ ]:
%%writefile rag_mcp_server.py

from mcp.server.fastmcp import FastMCP
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

mcp = FastMCP("RAG MCP Server")

documents = [
    {
        "id": "DOC001",
        "title": "Leave Policy",
        "content": "Employees receive 15 annual leave days and 10 sick leave days."
    },
    {
        "id": "DOC002",
        "title": "Work Hours Policy",
        "content": "Employees work from 9 AM to 6 PM Monday through Friday."
    },
    {
        "id": "DOC003",
        "title": "Payroll Policy",
        "content": "Salary is credited on the last working day of each month."
    }
]

doc_texts = [doc["content"] for doc in documents]
vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(doc_texts)

@mcp.tool()
def document_search(question: str):
    query_vector = vectorizer.transform([question])
    similarity_scores = cosine_similarity(
        query_vector,
        doc_vectors
    )[0]
    best_index = similarity_scores.argmax()
    best_doc = documents[best_index]
    confidence = float(similarity_scores[best_index])
    return {
        "question": question,
        "answer": best_doc["content"],
        "source_document": best_doc["title"],
        "document_id": best_doc["id"],
        "confidence_score": round(confidence, 3)
    }
print("RAG MCP Server Ready")

Writing rag_mcp_server.py


In [ ]:
from rag_mcp_server import document_search

RAG MCP Server Ready


In [ ]:
result = document_search("When is salary credited?")

print("Question:", result["question"])
print("Answer:", result["answer"])
print("Source Document:", result["source_document"])
print("Document ID:", result["document_id"])
print("Confidence Score:", result["confidence_score"])

Question: When is salary credited?
Answer: Salary is credited on the last working day of each month.
Source Document: Payroll Policy
Document ID: DOC003
Confidence Score: 0.522


Task 7: Build MCP-Powered HR Assistant

In [ ]:
%%writefile hr_assistant.py
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("HR Assistant")

employees = {
    "EMP001": {"name": "Ajay", "department": "Engineering"},
    "EMP002": {"name": "Priya", "department": "HR"}
}
leave_balances = {
    "EMP001": 12,
    "EMP002": 8
}
payroll_data = {
    "EMP001": {"salary": 50000, "month": "June"},
    "EMP002": {"salary": 45000, "month": "June"}
}
policies = {
    "leave": "Employees receive 15 annual leave days and 10 sick leave days.",
    "payroll": "Salary is credited on the last working day of every month.",
    "work hours": "Working hours are 9 AM to 6 PM."
}
onboarding_guide = """
1. Submit required documents.
2. Complete HR verification.
3. Receive employee ID.
4. Attend onboarding session.
5. Get system access.
"""
@mcp.tool()
def employee_lookup(employee_id: str):
    return employees.get(employee_id, "Employee not found")

@mcp.tool()
def leave_management(employee_id: str):
    return leave_balances.get(employee_id, "Employee not found")

@mcp.tool()
def payroll_search(employee_id: str):
    return payroll_data.get(employee_id, "Employee not found")

@mcp.tool()
def policy_search(topic: str):
    return policies.get(topic.lower(), "Policy not found")

@mcp.tool()
def onboarding_guidance():
    return onboarding_guide
def hr_assistant(query):
    query = query.lower()
    if "employee" in query:
        result = employee_lookup("EMP001")
        return {
            "tool_used": "Employee Lookup",
            "response": result
        }
    elif "leave" in query:
        result = leave_management("EMP001")
        return {
            "tool_used": "Leave Management",
            "response": result
        }
    elif "salary" in query or "payroll" in query:
        result = payroll_search("EMP001")
        return {
            "tool_used": "Payroll Search",
            "response": result
        }
    elif "policy" in query:
        result = policy_search("leave")
        return {
            "tool_used": "Policy Search",
            "response": result
        }
    elif "onboarding" in query:
        result = onboarding_guidance()
        return {
            "tool_used": "Onboarding Guidance",
            "response": result
        }
    return {
        "tool_used": "None",
        "response": "Unable to determine request."
    }
print("HR Assistant Ready")

Writing hr_assistant.py


In [ ]:
from hr_assistant import hr_assistant

HR Assistant Ready


In [ ]:
result = hr_assistant("Show employee details")

print("Tool Used:", result["tool_used"])
print("Response:", result["response"])

Tool Used: Employee Lookup
Response: {'name': 'Ajay', 'department': 'Engineering'}


In [ ]:
result = hr_assistant("Check leave balance")

print("Tool Used:", result["tool_used"])
print("Response:", result["response"])

Tool Used: Leave Management
Response: 12


In [ ]:
result = hr_assistant("Show payroll details")

print("Tool Used:", result["tool_used"])
print("Response:", result["response"])

Tool Used: Payroll Search
Response: {'salary': 50000, 'month': 'June'}


In [ ]:
result = hr_assistant("What is the leave policy?")

print("Tool Used:", result["tool_used"])
print("Response:", result["response"])

Tool Used: Leave Management
Response: 12


In [ ]:
result = hr_assistant("Explain onboarding process")

print("Tool Used:", result["tool_used"])
print("Response:")
print(result["response"])

Tool Used: Onboarding Guidance
Response:

1. Submit required documents.
2. Complete HR verification.
3. Receive employee ID.
4. Attend onboarding session.
5. Get system access.



Task 8: Enterprise AI Gateway

In [ ]:
%%writefile enterprise_gateway.py
import logging
from datetime import datetime

logging.basicConfig(
    filename="gateway.log",
    level=logging.INFO,
    format="%(asctime)s - %(message)s"
)
tool_registry = {
    "employee_lookup": "Employee MCP Server",
    "leave_management": "Leave MCP Server",
    "payroll_search": "Payroll MCP Server",
    "policy_search": "Policy MCP Server"
}
valid_users = {
    "admin": "admin123",
    "hr_user": "hr123"
}
permissions = {
    "admin": [
        "employee_lookup",
        "leave_management",
        "payroll_search",
        "policy_search"
    ],
    "hr_user": [
        "employee_lookup",
        "leave_management",
        "policy_search"
    ]
}
request_count = 0
def authenticate(username, password):
    if username in valid_users:
        return valid_users[username] == password
    return False
def authorize(username, tool_name):
    return tool_name in permissions.get(username, [])
def route_request(tool_name):
    return tool_registry.get(
        tool_name,
        "Tool Not Registered"
    )
def monitor_request():
    global request_count
    request_count += 1
    return request_count
def log_request(username, tool_name):
    logging.info(
        f"User={username} Tool={tool_name}"
    )
def enterprise_gateway(
    username,
    password,
    tool_name
):
    if not authenticate(
        username,
        password
    ):
        return {
            "status": "Failed",
            "message": "Authentication Failed"
        }
    if not authorize(
        username,
        tool_name
    ):
        return {
            "status": "Failed",
            "message": "Authorization Failed"
        }
    destination = route_request(
        tool_name
    )
    count = monitor_request()
    log_request(
        username,
        tool_name
    )
    return {
        "status": "Success",
        "user": username,
        "tool": tool_name,
        "routed_to": destination,
        "request_number": count,
        "timestamp": str(datetime.now())
    }
print("Enterprise AI Gateway Ready")

Writing enterprise_gateway.py


In [ ]:
from enterprise_gateway import enterprise_gateway

Enterprise AI Gateway Ready


In [ ]:
#Authentication Success
result = enterprise_gateway(
    "admin",
    "admin123",
    "payroll_search"
)
for key, value in result.items():
    print(f"{key}: {value}")

status: Success
user: admin
tool: payroll_search
routed_to: Payroll MCP Server
request_number: 3
timestamp: 2026-06-23 10:42:04.172964


In [ ]:
#Authentication Failure
result = enterprise_gateway(
    "admin",
    "wrongpassword",
    "payroll_search"
)
print(result)

{'status': 'Failed', 'message': 'Authentication Failed'}


Task 9: Build BlackRoth MCP Platform

In [ ]:
%%writefile blackroth_platform.py
from mcp.server.fastmcp import FastMCP
import logging
from datetime import datetime

mcp=FastMCP("BlackRoth Platform")
logging.basicConfig(
    filename="blackroth_audit.log",
    level = logging.INFO,
    format = "%(asctime)s-%(message)s"
)
def audit_log(user, action):
  logging.info(f"User={user} Action={action}")

user_roles = {
    "admin": "admin",
    "hr_user": "hr",
    "support_user": "support",
    "manager":"manager"
}
permissions = {
    "admin": [
        "hr_services",
        "payroll_services",
        "customer_support",
        "project_management",
        "documentation_search",
        "analytics"
    ],
    "hr": [
        "hr_services",
        "payroll_services"
    ],
    "support": [
        "customer_support",
        "documentation_search"
    ],
    "manager": [
        "project_management",
        "analytics"
    ]
}
def has_access(username, module):
    role = user_roles.get(username)
    if role is None:
        return False
    return module in permissions.get(role, [])
service_registry = {
    "hr_services": "HR Module",
    "payroll_services": "Payroll Module",
    "customer_support": "Support Module",
    "project_management": "Project Module",
    "documentation_search": "Knowledge Module",
    "analytics": "Analytics Module"
}
@mcp.tool()
def hr_services(username: str):
    if not has_access(username, "hr_services"):
        return {"error": "Access Denied"}
    audit_log(username, "HR_SERVICE")
    return {
        "employees": 250,
        "active_employees": 230
    }
@mcp.tool()
def payroll_services(username: str):
    if not has_access(username, "payroll_services"):
        return {"error": "Access Denied"}
    audit_log(username, "PAYROLL_SERVICE")
    return {
        "current_month_payroll": 500000,
        "processed": True
    }
@mcp.tool()
def customer_support(username: str):
    if not has_access(username, "customer_support"):
        return {"error": "Access Denied"}
    audit_log(username, "CUSTOMER_SUPPORT")
    return {
        "open_tickets": 18,
        "resolved_today": 42
    }
@mcp.tool()
def project_management(username: str):
    if not has_access(username, "project_management"):
        return {"error": "Access Denied"}
    audit_log(username, "PROJECT_MANAGEMENT")
    return {
        "active_projects": 12,
        "completed_projects": 35
    }
@mcp.tool()
def documentation_search(username: str, query: str):
    if not has_access(username, "documentation_search"):
        return {"error": "Access Denied"}
    audit_log(username, f"DOC_SEARCH:{query}")
    docs = {
        "leave policy":
            "Employees receive 15 annual leave days.",
        "payroll":
            "Salary is credited monthly.",
        "onboarding":
            "Complete HR forms before joining."
    }
    return {
        "query": query,
        "result": docs.get(
            query.lower(),
            "No document found"
        )
    }
@mcp.tool()
def analytics(username: str):
    if not has_access(username, "analytics"):
        return {"error": "Access Denied"}
    audit_log(username, "ANALYTICS")
    return {
        "monthly_revenue": 1200000,
        "growth_percent": 18
    }
@mcp.tool()
def platform_registry():
    return service_registry

if __name__ == "__main__":
    mcp.run()

Writing blackroth_platform.py


In [ ]:
from blackroth_platform import *
print(hr_services("admin"))

{'employees': 250, 'active_employees': 230}


In [ ]:
print(payroll_services("hr_user"))

{'current_month_payroll': 500000, 'processed': True}


In [ ]:
print(documentation_search(
    "support_user",
    "leave policy"
))

{'query': 'leave policy', 'result': 'Employees receive 15 annual leave days.'}


In [ ]:
print(analytics("manager"))

{'monthly_revenue': 1200000, 'growth_percent': 18}


In [ ]:
print(payroll_services("support_user"))

{'error': 'Access Denied'}


In [ ]:
registry = platform_registry()

for module, description in registry.items():
    print(f"{module} : {description}")

hr_services : HR Module
payroll_services : Payroll Module
customer_support : Support Module
project_management : Project Module
documentation_search : Knowledge Module
analytics : Analytics Module


Bonus Challenge - a BlackRoth Enterprise AI Platform

In [ ]:
from mcp.server.fastmcp import FastMCP
import logging
from datetime import datetime

mcp = FastMCP("BlackRoth Enterprise AI Platform")
logging.basicConfig(
    filename="enterprise_audit.log",
    level=logging.INFO
)
roles = {
    "admin": [
        "hr",
        "payroll",
        "support",
        "project",
        "analytics"
    ],
    "hr_user": ["hr"],
    "manager": ["project", "analytics"],
    "support_user": ["support"]
}
registry = {
    "HR Assistant": "Active",
    "Payroll Assistant": "Active",
    "Customer Support Agent": "Active",
    "Document Search": "Active",
    "Project Agent": "Active",
    "Analytics Assistant": "Active"
}
def authorize(user, service):
    return service in roles.get(user, [])

def audit(user, action):
    logging.info(
        f"{datetime.now()} | {user} | {action}"
    )
@mcp.tool()
def hr_assistant(user):

    if not authorize(user, "hr"):
        return "Access Denied"

    audit(user, "HR_ASSISTANT")

    return {
        "employees": 250,
        "leave_requests": 12
    }
@mcp.tool()
def payroll_assistant(user):

    if not authorize(user, "payroll"):
        return "Access Denied"

    audit(user, "PAYROLL")

    return {
        "processed": True,
        "monthly_payroll": 500000
    }
@mcp.tool()
def customer_support_agent(user):

    if not authorize(user, "support"):
        return "Access Denied"

    audit(user, "SUPPORT")

    return {
        "open_tickets": 18,
        "resolved_today": 42
    }
@mcp.tool()
def document_search(query):

    documents = {
        "leave policy":
        "Employees receive 15 annual leave days.",

        "payroll":
        "Salary is credited monthly."
    }

    return {
        "query": query,
        "result":
        documents.get(
            query.lower(),
            "No document found"
        ),
        "confidence": 0.91
    }
@mcp.tool()
def project_agent(user):

    if not authorize(user, "project"):
        return "Access Denied"

    audit(user, "PROJECT")

    return {
        "active_projects": 12,
        "completed_projects": 35
    }
@mcp.tool()
def analytics_assistant(user):

    if not authorize(user, "analytics"):
        return "Access Denied"

    audit(user, "ANALYTICS")

    return {
        "revenue": 1200000,
        "growth": "18%"
    }
@mcp.tool()
def platform_registry():
    return registry

print("BlackRoth Enterprise Platform Ready")

BlackRoth Enterprise Platform Ready


In [ ]:
#Platform Registry
registry = platform_registry()

for service, status in registry.items():
    print(f"{service}: {status}")

HR Assistant: Active
Payroll Assistant: Active
Customer Support Agent: Active
Document Search: Active
Project Agent: Active
Analytics Assistant: Active


In [ ]:
#HR Assistant
print(hr_assistant("admin"))

{'employees': 250, 'leave_requests': 12}


In [ ]:
#Payroll Assistant
print(payroll_assistant("admin"))

{'processed': True, 'monthly_payroll': 500000}


In [ ]:
#Customer Support Agent
print(customer_support_agent("admin"))

{'open_tickets': 18, 'resolved_today': 42}


In [ ]:
#Document Search
print(document_search("leave policy"))

{'query': 'leave policy', 'result': 'Employees receive 15 annual leave days.', 'confidence': 0.91}


In [ ]:
#Project Agent
print(project_agent("manager"))

{'active_projects': 12, 'completed_projects': 35}


In [ ]:
#Analytics Assistant
print(analytics_assistant("manager"))

{'revenue': 1200000, 'growth': '18%'}


In [ ]:
#Access Denied
print(payroll_assistant("support_user"))

Access Denied
